# 11 — Best Image Combinations (RelMS)

Tests the two most promising two-image combinations using the RelMS block distance:

| Combination | Blocks | X shape |
|-------------|--------|---------|
| `masked_tra + cor` | `p1=[110, 110]` | 416 × 220 |
| `masked_tra + gfc_tra` | `p1=[110, 110]` | 416 × 220 |

Both k=3 and k=4 are tested for each combination.

## Install

In [1]:
import subprocess, sys

def run_pip(*args):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet"] + list(args)
    r   = subprocess.run(cmd, capture_output=True, text=True)
    tag = args[-1]
    if r.returncode == 0:
        print(f"  OK    {tag}")
    else:
        print(f"  FAIL  {tag}")
        print(r.stderr[-400:] if r.stderr else "(no stderr)")

run_pip("robust-mixed-dist==0.1.26")
for pkg in ["kmedoids", "openpyxl"]:
    run_pip(pkg)
print("Done.")

import importlib.metadata
print(f"robust-mixed-dist version: {importlib.metadata.version('robust-mixed-dist')}")

  OK    robust-mixed-dist==0.1.26
  OK    kmedoids
  OK    openpyxl
Done.
robust-mixed-dist version: 0.1.26


## Imports

In [2]:
import os
import numpy as np
import pandas as pd

import kmedoids
# function lives in robust_mixed_dist.mixed — not at the package top level
from robust_mixed_dist.mixed import related_metric_scaling_dist_matrix_faster
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
)

print("All imports successful.")

All imports successful.


## Step 1 — Load PCA CSVs

Each file is 416 × 111 (patient_id + 110 PCA components).  
Drop `patient_id` to get the 416 × 110 feature matrix.

In [3]:
NB_FOLDER = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(NB_FOLDER, '..', '..'))
DATA_PIPELINE_DIR = os.path.join(PROJECT_ROOT, 'notebooks', '01_data_pipeline')
OUTPUT_DIR = NB_FOLDER
NB_DIR = DATA_PIPELINE_DIR      # legacy alias: shared pipeline inputs (pca_*.csv, clinical xlsx)
NB_PADDED = OUTPUT_DIR          # legacy alias: this notebook's own outputs

def load_pca(fname):
    df = pd.read_csv(os.path.join(NB_DIR, fname))
    ids = df["patient_id"].values
    X   = df.drop(columns=["patient_id"]).to_numpy(dtype=np.float64)
    return ids, X

masked_tra_ids, masked_tra_pca = load_pca("pca_masked_tra.csv")
cor_ids,        cor_pca        = load_pca("pca_cor.csv")
gfc_tra_ids,    gfc_tra_pca    = load_pca("pca_gfc_tra.csv")

print(f"pca_masked_tra : {masked_tra_pca.shape}")
print(f"pca_cor        : {cor_pca.shape}")
print(f"pca_gfc_tra    : {gfc_tra_pca.shape}")

# Confirm all three files share the same patient order
assert np.array_equal(masked_tra_ids, cor_ids),     "Patient order mismatch: masked_tra vs cor"
assert np.array_equal(masked_tra_ids, gfc_tra_ids), "Patient order mismatch: masked_tra vs gfc_tra"
patient_ids = masked_tra_ids
print(f"\nPatient IDs aligned across all 3 files — {len(patient_ids)} patients.")

pca_masked_tra : (416, 110)
pca_cor        : (416, 110)
pca_gfc_tra    : (416, 110)

Patient IDs aligned across all 3 files — 416 patients.


## Load CDR for evaluation

In [4]:
df_clin = pd.read_excel(
    os.path.join(NB_DIR, "oasis_cross-sectional-5708aa0a98d82080.xlsx")
)
if "ID" in df_clin.columns:
    df_clin = df_clin.rename(columns={"ID": "patient_id"})

df_clin["patient_id"] = df_clin["patient_id"].str.replace(r"_MR\d+$", "", regex=True)
df_clin = (
    df_clin
    .sort_values("CDR", na_position="last")
    .drop_duplicates(subset="patient_id", keep="first")
    .reset_index(drop=True)
)

cdr_series = df_clin.set_index("patient_id")["CDR"].reindex(patient_ids)
mask_cdr   = cdr_series.notnull().values
cdr_int    = (cdr_series.values[mask_cdr] * 2).astype(int)

print(f"Patients with CDR : {mask_cdr.sum()} / {len(patient_ids)}")
print(f"CDR categories    : {sorted(set(cdr_int))}  (CDR × 2)")

Patients with CDR : 235 / 416
CDR categories    : [np.int64(0), np.int64(1), np.int64(2), np.int64(4)]  (CDR × 2)


## Step 2 — Build feature matrices and compute distance matrices

Each combination is horizontally concatenated into a 416 × 220 matrix.  
`p1=[110, 110]` tells RelMS to treat the two blocks separately and normalise
each by its own geometric variability before combining.

The function receives a DataFrame to avoid a bug in 0.1.26 where numpy arrays
leave `X_arr` unassigned.

In [5]:
def compute_relms(X, label):
    print(f"  [{label}] computing RelMS on X={X.shape} ...", end=" ", flush=True)
    D = related_metric_scaling_dist_matrix_faster(
        X=pd.DataFrame(X),   # DataFrame required — avoids numpy bug in 0.1.26
        p1=[110, 110],
        p2=0,
        p3=0,
        d1="euclidean",
        d2=None,
        d3=None,
        weights=None,
    )
    nan_count = np.isnan(D).sum()
    print(f"done  shape={D.shape}  range=[{D.min():.4f}, {D.max():.4f}]  "
          f"NaN={nan_count} {'OK' if nan_count == 0 else 'WARNING'}")
    return D

# Combination 1: masked_tra + cor
X1 = np.hstack([masked_tra_pca, cor_pca])         # 416 × 220
D1 = compute_relms(X1, "masked_tra + cor")

# Combination 2: masked_tra + gfc_tra
X2 = np.hstack([masked_tra_pca, gfc_tra_pca])     # 416 × 220
D2 = compute_relms(X2, "masked_tra + gfc_tra")

  [masked_tra + cor] computing RelMS on X=(416, 220) ... done  shape=(416, 416)  range=[0.0000, 2.6271]  NaN=0 OK
  [masked_tra + gfc_tra] computing RelMS on X=(416, 220) ... done  shape=(416, 416)  range=[0.0000, 2.5517]  NaN=0 OK


## Steps 3 & 4 — Cluster and evaluate

Run `kmedoids.fasterpam` on each distance matrix for k=3 and k=4,
then compute CDR crosstab, spread, silhouette, ARI and NMI.

In [6]:
COMBINATIONS = [
    ("masked_tra+cor",     D1),
    ("masked_tra+gfc_tra", D2),
]

all_results = []

for combo_name, D in COMBINATIONS:
    print(f"\n{'='*60}")
    print(f"  {combo_name}")
    print(f"{'='*60}")

    for k in [3, 4]:
        result = kmedoids.fasterpam(D, medoids=k, random_state=42)
        labels = np.array(result.labels, dtype=int)

        sizes = {c: int((labels == c).sum()) for c in sorted(set(labels))}
        print(f"\n  k={k}  loss={result.loss:.4f}  sizes={sizes}")

        # ── CDR crosstab ──────────────────────────────────────────────────────
        df_ev = pd.DataFrame({"cluster": labels, "CDR": cdr_series.values})
        df_ev = df_ev[df_ev["CDR"].notnull()].copy()
        df_ev["CDR"] = df_ev["CDR"].astype(str)

        ct = pd.crosstab(df_ev["cluster"], df_ev["CDR"],
                         margins=True, margins_name="Total")
        print("  Count:")
        print(ct.to_string())

        pct = pd.crosstab(df_ev["cluster"], df_ev["CDR"],
                          normalize="index").mul(100).round(1)
        print("\n  Row % (CDR proportion within each cluster):")
        print(pct.to_string())

        # ── Spread ────────────────────────────────────────────────────────────
        pct_raw = pd.crosstab(df_ev["cluster"], df_ev["CDR"], normalize="index").mul(100)
        pct_h   = pct_raw["0.0"] if "0.0" in pct_raw.columns \
                  else pd.Series(0.0, index=pct_raw.index)
        spread  = float(pct_h.max() - pct_h.min())

        # ── Silhouette ────────────────────────────────────────────────────────
        sil = silhouette_score(D, labels, metric="precomputed")

        # ── ARI / NMI ─────────────────────────────────────────────────────────
        ari = adjusted_rand_score(cdr_int, labels[mask_cdr])
        nmi = normalized_mutual_info_score(cdr_int, labels[mask_cdr])

        print(f"\n  spread={spread:.2f}%  sil={sil:.4f}  ARI={ari:.4f}  NMI={nmi:.4f}")

        all_results.append({
            "combination":        combo_name,
            "k":                  k,
            "n_patients":         len(patient_ids),
            "silhouette":         round(sil, 4),
            "ari":                round(ari, 4),
            "nmi":                round(nmi, 4),
            "spread_pct_healthy": round(spread, 2),
        })

print("\n\nAll combinations complete.")


  masked_tra+cor

  k=3  loss=727.4207  sizes={np.int64(0): 217, np.int64(1): 107, np.int64(2): 92}
  Count:
CDR      0.0  0.5  1.0  2.0  Total
cluster                           
0         58   26    5    0     89
1         47   38   19    2    106
2         30    6    4    0     40
Total    135   70   28    2    235

  Row % (CDR proportion within each cluster):
CDR       0.0   0.5   1.0  2.0
cluster                       
0        65.2  29.2   5.6  0.0
1        44.3  35.8  17.9  1.9
2        75.0  15.0  10.0  0.0

  spread=30.66%  sil=0.0068  ARI=-0.0028  NMI=0.0444

  k=4  loss=720.7240  sizes={np.int64(0): 190, np.int64(1): 76, np.int64(2): 61, np.int64(3): 89}
  Count:
CDR      0.0  0.5  1.0  2.0  Total
cluster                           
0         45   15    4    0     64
1         37   26   11    1     75
2         24   25    9    1     59
3         29    4    4    0     37
Total    135   70   28    2    235

  Row % (CDR proportion within each cluster):
CDR       0.0   0.5   1.

## Step 5 — Summary table and save

In [7]:
df_results = pd.DataFrame(all_results)

print("RelMS — best two-image combinations")
print("=" * 65)
print(df_results.to_string(index=False))
print(f"\nEvaluated on {mask_cdr.sum()} patients with CDR (ARI/NMI) "
      f"and all {len(patient_ids)} patients (silhouette).")

out_path = os.path.join(NB_PADDED, "results_best_image_combinations.csv")
df_results.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

RelMS — best two-image combinations
       combination  k  n_patients  silhouette     ari    nmi  spread_pct_healthy
    masked_tra+cor  3         416      0.0068 -0.0028 0.0444               30.66
    masked_tra+cor  4         416      0.0086  0.0141 0.0437               37.70
masked_tra+gfc_tra  3         416      0.0065 -0.0287 0.1009               48.05
masked_tra+gfc_tra  4         416      0.0057 -0.0359 0.0624               40.60

Evaluated on 235 patients with CDR (ARI/NMI) and all 416 patients (silhouette).

Saved: C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data\notebooks\new_results_with_padding\results_best_image_combinations.csv


---

## Image + Clinical Combinations\n",
    "\n",
    "Same two image combinations but each combined with clinical variables.\n",
    "Missing values in Educ, SES, MMSE are **dropped** — no imputation.\n",
    "\n",
    "| Column group | Columns | Block |\n",
    "|---|---|---|\n",
    "| Image block 1 | masked_tra — 110 PCA | `p1[0]=110` |\n",
    "| Image block 2 | cor or gfc_tra — 110 PCA | `p1[1]=110` |\n",
    "| Clinical quant block | Age, MMSE, eTIV, nWBV, ASF | `p1[2]=5` |\n",
    "| Binary | M/F | `p2=1` |\n",
    "| Multiclass | Educ, SES | `p3=2` |\n",
    "\n",
    "Final X shape per combination: **(n_patients, 228)**.

### Step 1 — Load and filter clinical data

In [8]:
df_clin_full = pd.read_excel(
    os.path.join(NB_DIR, "oasis_cross-sectional-5708aa0a98d82080.xlsx")
)
if "ID" in df_clin_full.columns:
    df_clin_full = df_clin_full.rename(columns={"ID": "patient_id"})

df_clin_full["patient_id"] = df_clin_full["patient_id"].str.replace(r"_MR\d+$", "", regex=True)
df_clin_full = (
    df_clin_full
    .sort_values("CDR", na_position="last")
    .drop_duplicates(subset="patient_id", keep="first")
    .reset_index(drop=True)
)

FEATURE_COLS = ["Age", "M/F", "Educ", "SES", "MMSE", "eTIV", "nWBV", "ASF"]
KEEP         = ["patient_id"] + FEATURE_COLS + ["CDR"]
df_clin_full = df_clin_full[KEEP].copy()

n_before     = len(df_clin_full)
df_complete  = df_clin_full.dropna(subset=["Educ", "SES", "MMSE"]).reset_index(drop=True)
n_after      = len(df_complete)

print(f"Patients before dropping : {n_before}")
print(f"Patients after  dropping : {n_after}  (dropped {n_before - n_after})")
print(f"Patients with CDR        : {df_complete['CDR'].notnull().sum()} / {n_after}")

Patients before dropping : 416
Patients after  dropping : 216  (dropped 200)
Patients with CDR        : 216 / 216


### Step 2 — Filter PCA matrices to complete patients

In [9]:
def filter_pca(all_pca, all_ids, complete_df):
    df_tmp = pd.DataFrame(all_pca)
    df_tmp.insert(0, "patient_id", all_ids)
    aligned = complete_df[["patient_id"]].merge(df_tmp, on="patient_id", how="left")
    return aligned.drop(columns=["patient_id"]).to_numpy(dtype=np.float64)

masked_c  = filter_pca(masked_tra_pca, masked_tra_ids, df_complete)
cor_c     = filter_pca(cor_pca,        cor_ids,        df_complete)
gfc_tra_c = filter_pca(gfc_tra_pca,    gfc_tra_ids,    df_complete)

print(f"masked_tra (complete) : {masked_c.shape}  NaN={np.isnan(masked_c).sum()}")
print(f"cor        (complete) : {cor_c.shape}  NaN={np.isnan(cor_c).sum()}")
print(f"gfc_tra    (complete) : {gfc_tra_c.shape}  NaN={np.isnan(gfc_tra_c).sum()}")

masked_tra (complete) : (216, 110)  NaN=0
cor        (complete) : (216, 110)  NaN=0
gfc_tra    (complete) : (216, 110)  NaN=0


### Step 3 — Build feature matrices (image + clinical)

In [10]:
dc = df_complete.copy()
dc["MF"]   = (dc["M/F"] == "M").astype(int)
dc["Educ"] = dc["Educ"].astype(int)
dc["SES"]  = dc["SES"].astype(int)

QUANT_CLIN = ["Age", "MMSE", "eTIV", "nWBV", "ASF"]
clin_quant = dc[QUANT_CLIN].to_numpy(dtype=np.float64)   # (n, 5)
clin_bin   = dc[["MF"]].to_numpy(dtype=np.float64)       # (n, 1)
clin_multi = dc[["Educ", "SES"]].to_numpy(dtype=np.float64)  # (n, 2)

# Column order: [image block 1] + [image block 2] + [quant clinical] + [binary] + [multiclass]
XC1 = np.hstack([masked_c,  cor_c,     clin_quant, clin_bin, clin_multi])  # masked_tra + cor + clinical
XC2 = np.hstack([masked_c,  gfc_tra_c, clin_quant, clin_bin, clin_multi])  # masked_tra + gfc_tra + clinical

print(f"XC1 (masked_tra+cor+clinical)     : {XC1.shape}  — expected ({n_after}, 228)")
print(f"XC2 (masked_tra+gfc_tra+clinical) : {XC2.shape}  — expected ({n_after}, 228)")
assert XC1.shape[1] == 228, f"Column count mismatch: {XC1.shape[1]}"
assert XC2.shape[1] == 228, f"Column count mismatch: {XC2.shape[1]}"

# CDR aligned to complete patients
cdr_c      = dc["CDR"].values
mask_cdr_c = pd.Series(cdr_c).notnull().values
cdr_int_c  = (cdr_c[mask_cdr_c] * 2).astype(int)
pids_c     = dc["patient_id"].values

print(f"Patients with CDR : {mask_cdr_c.sum()} / {len(pids_c)} "
      f"— {'all have CDR' if mask_cdr_c.all() else f'{(~mask_cdr_c).sum()} missing CDR'}")

XC1 (masked_tra+cor+clinical)     : (216, 228)  — expected (216, 228)
XC2 (masked_tra+gfc_tra+clinical) : (216, 228)  — expected (216, 228)
Patients with CDR : 216 / 216 — all have CDR


### Step 4 — Compute distance matrices (p1=[110, 110, 5])

In [11]:
def compute_relms_clinical(X, label):
    print(f"  [{label}] computing RelMS on X={X.shape} ...", end=" ", flush=True)
    D = related_metric_scaling_dist_matrix_faster(
        X=pd.DataFrame(X),
        p1=[110, 110, 5],   # image block 1 + image block 2 + clinical quant block
        p2=1,               # M/F binary
        p3=2,               # Educ, SES multiclass
        d1="euclidean",
        d2="sokal",
        d3="hamming",
        weights=None,
    )
    nan_count = np.isnan(D).sum()
    print(f"done  shape={D.shape}  range=[{D.min():.4f}, {D.max():.4f}]  "
          f"NaN={nan_count} {'OK' if nan_count == 0 else 'WARNING'}")
    return D

DC1 = compute_relms_clinical(XC1, "masked_tra+cor+clinical")
DC2 = compute_relms_clinical(XC2, "masked_tra+gfc_tra+clinical")

  [masked_tra+cor+clinical] computing RelMS on X=(216, 228) ... done  shape=(216, 216)  range=[0.0000, 8.5261]  NaN=0 OK
  [masked_tra+gfc_tra+clinical] computing RelMS on X=(216, 228) ... done  shape=(216, 216)  range=[0.0000, 8.5291]  NaN=0 OK


C:\Users\aregk\AppData\Local\Programs\Python\Python312\Lib\site-packages\robust_mixed_dist\mixed.py:1002: UserWarning: Gram matrix for block 5 is not PSD (min eig=-1.47e+01). Transformation applied.
  warnings.warn(
C:\Users\aregk\AppData\Local\Programs\Python\Python312\Lib\site-packages\robust_mixed_dist\mixed.py:1002: UserWarning: Gram matrix for block 5 is not PSD (min eig=-1.47e+01). Transformation applied.
  warnings.warn(


### Steps 5 & 6 — Cluster and evaluate

In [12]:
CLIN_COMBOS = [
    ("masked_tra+cor+clinical",     DC1),
    ("masked_tra+gfc_tra+clinical", DC2),
]

clinical_results = []

for combo_name, D in CLIN_COMBOS:
    print(f"\n{'='*60}")
    print(f"  {combo_name}")
    print(f"{'='*60}")

    for k in [3, 4]:
        result = kmedoids.fasterpam(D, medoids=k, random_state=42)
        labels = np.array(result.labels, dtype=int)

        sizes = {c: int((labels == c).sum()) for c in sorted(set(labels))}
        print(f"\n  k={k}  loss={result.loss:.4f}  sizes={sizes}")

        # ── CDR crosstab ──────────────────────────────────────────────────────
        df_ev = pd.DataFrame({"cluster": labels, "CDR": cdr_c})
        df_ev = df_ev[df_ev["CDR"].notnull()].copy()
        df_ev["CDR"] = df_ev["CDR"].astype(str)

        ct  = pd.crosstab(df_ev["cluster"], df_ev["CDR"],
                          margins=True, margins_name="Total")
        pct = pd.crosstab(df_ev["cluster"], df_ev["CDR"],
                          normalize="index").mul(100).round(1)
        print("  Count:")
        print(ct.to_string())
        print("\n  Row %:")
        print(pct.to_string())

        # ── Spread ────────────────────────────────────────────────────────────
        pct_raw = pd.crosstab(df_ev["cluster"], df_ev["CDR"], normalize="index").mul(100)
        pct_h   = pct_raw["0.0"] if "0.0" in pct_raw.columns \
                  else pd.Series(0.0, index=pct_raw.index)
        spread  = float(pct_h.max() - pct_h.min())

        # ── Silhouette / ARI / NMI ────────────────────────────────────────────
        sil = silhouette_score(D, labels, metric="precomputed")
        ari = adjusted_rand_score(cdr_int_c, labels[mask_cdr_c])
        nmi = normalized_mutual_info_score(cdr_int_c, labels[mask_cdr_c])

        print(f"\n  spread={spread:.2f}%  sil={sil:.4f}  ARI={ari:.4f}  NMI={nmi:.4f}")

        clinical_results.append({
            "combination":        combo_name,
            "k":                  k,
            "n_patients":         len(pids_c),
            "silhouette":         round(sil, 4),
            "ari":                round(ari, 4),
            "nmi":                round(nmi, 4),
            "spread_pct_healthy": round(spread, 2),
        })

print("\n\nAll clinical combinations complete.")


  masked_tra+cor+clinical

  k=3  loss=1381.9747  sizes={np.int64(0): 73, np.int64(1): 70, np.int64(2): 73}
  Count:
CDR      0.0  0.5  1.0  2.0  Total
cluster                           
0         39   21   12    1     73
1         36   26    7    1     70
2         58   10    5    0     73
Total    133   57   24    2    216

  Row %:
CDR       0.0   0.5   1.0  2.0
cluster                       
0        53.4  28.8  16.4  1.4
1        51.4  37.1  10.0  1.4
2        79.5  13.7   6.8  0.0

  spread=28.02%  sil=0.0200  ARI=0.0323  NMI=0.0422

  k=4  loss=1369.6661  sizes={np.int64(0): 33, np.int64(1): 72, np.int64(2): 71, np.int64(3): 40}
  Count:
CDR      0.0  0.5  1.0  2.0  Total
cluster                           
0         16   12    4    1     33
1         57   10    5    0     72
2         38   20   12    1     71
3         22   15    3    0     40
Total    133   57   24    2    216

  Row %:
CDR       0.0   0.5   1.0  2.0
cluster                       
0        48.5  36.4  12.1  3.

### Step 7 — Append to existing CSV with `includes_clinical` column

In [13]:
out_path = os.path.join(NB_PADDED, "results_best_image_combinations.csv")

# Load existing image-only results, tag them
df_image_only = pd.read_csv(out_path)
df_image_only["includes_clinical"] = False

# Tag new clinical results
df_clinical = pd.DataFrame(clinical_results)
df_clinical["includes_clinical"] = True

# Combine and save
df_all = pd.concat([df_image_only, df_clinical], ignore_index=True)
df_all.to_csv(out_path, index=False)

print("Full results table (image-only + image+clinical):")
print("=" * 75)
print(df_all.to_string(index=False))
print(f"\nSaved {len(df_all)} rows to: {out_path}")

Full results table (image-only + image+clinical):
                combination  k  n_patients  silhouette     ari    nmi  spread_pct_healthy  includes_clinical
             masked_tra+cor  3         416      0.0068 -0.0028 0.0444               30.66              False
             masked_tra+cor  4         416      0.0086  0.0141 0.0437               37.70              False
         masked_tra+gfc_tra  3         416      0.0065 -0.0287 0.1009               48.05              False
         masked_tra+gfc_tra  4         416      0.0057 -0.0359 0.0624               40.60              False
    masked_tra+cor+clinical  3         216      0.0200  0.0323 0.0422               28.02               True
    masked_tra+cor+clinical  4         216      0.0138  0.0409 0.0414               30.68               True
masked_tra+gfc_tra+clinical  3         216      0.0203  0.0157 0.0201               18.20               True
masked_tra+gfc_tra+clinical  4         216      0.0138  0.0220 0.0411         